###  Pydantic

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)
model

ChatGroq(output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000002A575DABFE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A575E2C860>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from pydantic import BaseModel, Field

class movie(BaseModel):
    title:str= Field(description ='The title of the movie')
    year:int = Field(description='This is the movie was released')
    director:str= Field(description='Who is the diector for the movie ')
    rating:float= Field(description="The movies rating of the movie")

In [6]:
model_with_structured = model.with_structured_output(movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000002A575DABFE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A575E2C860>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This is the movie was released', 'type': 'integer'}, 'director': {'description': 'Who is the diector for the movie ', 'type': 'string'}, 'rating': {'description': 'The movies rating of the movie', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'movie'

In [8]:
model.invoke("Provide the movie of RRR")

AIMessage(content='You\'re referring to the Indian action-drama film "RRR" (Rise Roar Revolt). Here\'s a brief summary:\n\n**Movie Title:** RRR (Rise Roar Revolt)\n**Release Date:** March 25, 2022\n**Director:** S. S. Rajamouli\n**Cast:** N. T. Rama Rao Jr., Ram Charan, Alia Bhatt, Olivia Morris, and Ajay Devgn\n**Genre:** Action, Drama, Historical Fiction\n**Language:** Telugu (with dubbed versions in other languages)\n\n**Plot:**\n\nThe movie is set in the 1920s during the British colonial era in India. It\'s a fictional story based on the lives of two Indian revolutionaries, Alluri Sitarama Raju (played by Ram Charan) and Komaram Bheem (played by N. T. Rama Rao Jr.), who fought against the British Raj.\n\nThe film follows the story of Raju, a police officer, and Bheem, a tribal leader, who become unlikely friends and join forces to take down the British colonial powers. Along the way, they face numerous challenges, including betrayal, love, and sacrifice.\n\n**Awards and Accolades:*

In [7]:
model_with_structured.invoke("Provide the movie of RRR")

movie(title='RRR', year=2022, director='S. S. Rajamouli', rating=8.1)

In [14]:
### Message Output along side Parsed structures
from pydantic import BaseModel, Field

class movie(BaseModel):
    title:str= Field(..., description ='The title of the movie')
    year:int = Field(...,description='This is the movie was released')
    director:str= Field(...,description='Who is the diector for the movie ')
    rating:float= Field(...,description="The movies rating of the movie")

model_with_structured = model.with_structured_output(movie, include_raw = True)
response = model_with_structured.invoke("Provide the movie of RRR")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'f9nj7k1py', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8,"title":"RRR","year":2022}', 'name': 'movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 280, 'total_tokens': 318, 'completion_time': 0.092557725, 'completion_tokens_details': None, 'prompt_time': 0.015660411, 'prompt_tokens_details': None, 'queue_time': 0.088313932, 'total_time': 0.108218136}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed203-2c50-78b2-83ee-3caa07c7ecaf-0', tool_calls=[{'name': 'movie', 'args': {'director': 'S. S. Rajamouli', 'rating': 8, 'title': 'RRR', 'year': 2022}, 'id': 'f9nj7k1py', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 38, 'total_tokens': 318}),
 'parsed': movie(title

In [16]:
from pydantic import BaseModel

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float

structured_model = model.with_structured_output(MovieDetails)

response = structured_model.invoke(
    "Provide details about the movie RRR"
)

print(response)
print(response.title)

title='RRR' year=2022 cast=[Actor(name='N.T. Rama Rao Jr.', role='Komaram Bheem'), Actor(name='Ram Charan', role='Alluri Sitarama Raju')] genres=['Action', 'Drama'] budget=550000000.0
RRR


## Type Dict

In [18]:
from typing_extensions import TypedDict, Annotated

class movieDict(TypedDict):
    """ A Movie details"""
    tite:Annotated[str, ...,"The title of the movie"]
    year: Annotated[int, ..., "The year of the movie was released"]
    director:Annotated[str, ...," The Director of the movie "]
    rating:Annotated[float,..., "The movie rating out of 10"]

model_with_tyed_Dict = model.with_structured_output(movieDict)
model_with_tyed_Dict.invoke("please provide bahubali")


{'director': 'S. S. Rajamouli',
 'rating': 8.1,
 'tite': 'Baahubali',
 'year': 2015}

## Data Classes

In [26]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "John Doe's email is john.doe@gmail.com and phone number is +1-555-123-4567"
            }
        ]
    }
)

print(response)



{'messages': [HumanMessage(content="John Doe's email is john.doe@gmail.com and phone number is +1-555-123-4567", additional_kwargs={}, response_metadata={}, id='eafbd4c2-9078-40c8-9432-2edf706d2ff1'), AIMessage(content='{"name":"John Doe","email":"john.doe@gmail.com","phone":"+1-555-123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 207, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrUv6K71VYLwMQHGRdHa0uBsOeObn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ed22c-cedf-7d71-8041-2cd5e59fa15a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 207, '

In [25]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "John Doe's email is john.doe@gmail.com and phone number is +1-555-123-4567"
            }
        ]
    }
)

print(response["structured_response"])

name='John Doe' email='john.doe@gmail.com' phone='+1-555-123-4567'


In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of the person."""
    
    name: str
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "John Doe's email is john.doe@gmail.com and phone number is +1-555-123-4567"
            }
        ]
    }
)

contact = response["structured_response"]

print(contact)
print(contact.name)
print(contact.email)
print(contact.phone)

ContactInfo(name='John Doe', email='john.doe@gmail.com', phone='+1-555-123-4567')
John Doe
john.doe@gmail.com
+1-555-123-4567
